# 03 — Model Training

This notebook trains and compares three models:
1. **ResNet-50** (primary) — two-phase fine-tuning
2. **VGG-16** — transfer learning baseline
3. **ResNet-50 from scratch** — no-pretrain baseline

Evaluation metric: **Quadratic Weighted Kappa (QWK)**

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.utils.config import load_config, get_device
from src.data import get_train_transforms, get_val_transforms
from src.data.dataset import create_data_loaders
from src.models import build_resnet50, build_vgg16, Trainer

cfg    = load_config('../configs/default.yaml')
device = get_device()
print(f'Using device: {device}')

## 1. Data Loading

In [ ]:
train_loader, val_loader, test_loader = create_data_loaders(
    train_csv=cfg['paths']['train_csv'],
    val_csv=cfg['paths']['val_csv'],
    test_csv=cfg['paths']['test_csv'],
    train_image_dir=cfg['paths']['train_images'],
    val_image_dir=cfg['paths']['val_images'],
    test_image_dir=cfg['paths']['test_images'],
    train_transform=get_train_transforms(cfg['data']['image_size']),
    val_transform=get_val_transforms(cfg['data']['image_size']),
    image_size=cfg['data']['image_size'],
    batch_size=cfg['training']['batch_size'],
)
print(f'Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}')

## 2. ResNet-50 Training (Primary Model)

In [ ]:
model_r50 = build_resnet50(
    num_classes=5, fc_hidden=512, dropout=0.3,
    pretrained=True, unfreeze_blocks=2,
)
# Print parameter count
total   = sum(p.numel() for p in model_r50.parameters())
trainable = sum(p.numel() for p in model_r50.parameters() if p.requires_grad)
print(f'Total params: {total:,} | Trainable: {trainable:,} ({100*trainable/total:.1f}%)')

In [ ]:
trainer_r50 = Trainer(model_r50, device, cfg, save_dir='../outputs/models')
history_r50 = trainer_r50.train(train_loader, val_loader)
print('ResNet-50 training complete!')

## 3. Training Curves

In [ ]:
from src.utils.visualization import plot_training_curves

plot_training_curves(history_r50, '../outputs/figures/training_curves_r50.png')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_r50['train_loss'], label='Train Loss')
axes[0].plot(history_r50['val_loss'], label='Val Loss')
axes[0].set_title('Loss', fontweight='bold')
axes[0].legend()

axes[1].plot(history_r50['train_qwk'], label='Train QWK')
axes[1].plot(history_r50['val_qwk'], label='Val QWK')
axes[1].set_title('Quadratic Weighted Kappa', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best Val QWK: {max(history_r50["val_qwk"]):.4f}')

## 4. VGG-16 Baseline Comparison

Train VGG-16 on the same data for comparison.

In [ ]:
from src.models import build_vgg16

model_vgg = build_vgg16(num_classes=5, fc_hidden=512, dropout=0.3, pretrained=True)
trainer_vgg = Trainer(model_vgg, device, cfg, save_dir='../outputs/models')
# Reduce epochs for quick baseline
cfg_quick = dict(cfg)
cfg_quick['training'] = dict(cfg['training'])
cfg_quick['training']['epochs'] = 15
history_vgg = trainer_vgg.train(train_loader, val_loader)
print('VGG-16 training complete!')